In [1]:
from dotenv import load_dotenv
import os
import requests
import pandas as pd
import time
import json

In [2]:
load_dotenv()  
def get_token(secret: str, url: str = "https://dep.simondg.com/auth/login"):
    response = requests.post(url, json={"secret": secret})
    response.raise_for_status() # Raises an error if the request fails
    return response.json()["access_token"]


def get_students_by_subgroup(token: str, subgroup_id: int):
    url = f"https://dep.simondg.com/students/{subgroup_id}"
    headers = {"Authorization": f"Bearer {token}"}
    
    response = requests.get(url, headers=headers)
    response.raise_for_status()  # Raises an error if the request fails
    
    data = response.json()
    
    # Save JSON file named after the subgroup_id
    filename = f"{subgroup_id}.json"
    with open(filename, "w") as file:
        json.dump(data, file, indent=4)
    
    return data


secret = os.getenv("SECRET")
token = get_token(secret)



In [3]:
import os
import json
import pandas as pd

# Path to your CSV file
subgroup_file = "../data/unique_classgroups.csv"
df_subgroups = pd.read_csv(subgroup_file, header=None, names=['subgroup_id'])

all_data_json = {"students": []}

def clean_json(data):
    """
    Recursively remove 'count' keys and return cleaned data.
    """
    if isinstance(data, dict):
        return {k: clean_json(v) for k, v in data.items() if k != "count"}
    elif isinstance(data, list):
        return [clean_json(item) for item in data]
    else:
        return data

for subgroup_id in df_subgroups['subgroup_id'].dropna():
    if str(subgroup_id).startswith("EK"):
        continue

    json_filename = f"../data/classgroups/{subgroup_id}.json"

    # Load or fetch subgroup data
    if os.path.exists(json_filename):
        with open(json_filename, "r") as f:
            data = json.load(f)
    else:
        print(f"Fetching data for subgroup {subgroup_id}")
        data = get_students_by_subgroup(token, str(subgroup_id))
        with open(json_filename, "w") as f:
            json.dump(data, f, indent=4)

    # 🧹 Normalize possible structures
    if isinstance(data, dict) and "students" in data and isinstance(data["students"], list):
        students = data["students"]
    elif isinstance(data, dict) and "students" in data and isinstance(data["students"], dict):
        students = data["students"].get("students", [])
    elif isinstance(data, list):
        students = data
    else:
        print(f"⚠️ Unexpected JSON structure in {json_filename}, skipping.")
        continue

    # 🧼 Clean out 'count' keys
    students = clean_json(students)

    # Add to combined dataset
    all_data_json["students"].extend(students)

# Save combined JSON (cleaned)
combined_json_path = "../data/classgroups/all_students.json"
with open(combined_json_path, "w") as f:
    json.dump(all_data_json, f, indent=4)

print("✅ All done — cleaned and combined JSON saved to ../data/classgroups/all_students.json")


✅ All done — cleaned and combined JSON saved to ../data/classgroups/all_students.json


In [4]:

# Path to your JSON file
json_file = "../data/classgroups/all_students.json"  # replace with your actual file

# Load JSON data
with open(json_file, "r") as f:
    data = json.load(f)

# Extract the 'students' list
students_list = data.get("students", [])

# Convert to DataFrame
students_df = pd.DataFrame(students_list)

# Save to CSV if needed
students_df.to_csv("../data/all_students.csv", index=False)

students_df.to_csv("../data/classgroups/all_students.csv", index=False)

print(students_df.head())

   SUBGROEPID   SUBGROEPCODE  DEELGROEPID  \
0     5796255  PBA-TIN-TI/2F        26568   
1     5796255  PBA-TIN-TI/2F        26568   
2     5796255  PBA-TIN-TI/2F        26568   
3     5796255  PBA-TIN-TI/2F        26568   
4     5796255  PBA-TIN-TI/2F        26568   

                                                NAAM  \
0  u_6120838b45f8a23134fffa630b60c97ee0866614c6fd...   
1  u_085aeb527b4e8d54b80875da794daab58e1fae6e5a27...   
2  u_c61b2eb037f8853c426fa7516e8574a1a6f3fd923f86...   
3  u_4d234e0bc35f65a8e5f9f02b87316a1e9fb86803f960...   
4  u_1be2449101670229b6741ff0a97d8439113c71f31530...   

                                               EMAIL  
0  u_cacce4ad668d5089be6da28a29870572c9896c6e47f0...  
1  u_828e1f1daf3c4389d7f7a08d6576e906b1df96f4d58a...  
2  u_0dada03958fdd8bb38434971b53cce4eff74ba7a1b17...  
3  u_1e587def4acd0a69ce7cbfe5cfc99fd62510d1719c93...  
4  u_2f470606820175c7e14de69708bdbba617e582e998d7...  


In [5]:
students_df.head(10)

,SUBGROEPID,SUBGROEPCODE,DEELGROEPID,NAAM,EMAIL
0,5796255,PBA-TIN-TI/2F,26568,u_6120838b45f8a23134fffa630b60c97ee0866614c6fd...,u_cacce4ad668d5089be6da28a29870572c9896c6e47f0...
1,5796255,PBA-TIN-TI/2F,26568,u_085aeb527b4e8d54b80875da794daab58e1fae6e5a27...,u_828e1f1daf3c4389d7f7a08d6576e906b1df96f4d58a...
2,5796255,PBA-TIN-TI/2F,26568,u_c61b2eb037f8853c426fa7516e8574a1a6f3fd923f86...,u_0dada03958fdd8bb38434971b53cce4eff74ba7a1b17...
3,5796255,PBA-TIN-TI/2F,26568,u_4d234e0bc35f65a8e5f9f02b87316a1e9fb86803f960...,u_1e587def4acd0a69ce7cbfe5cfc99fd62510d1719c93...
4,5796255,PBA-TIN-TI/2F,26568,u_1be2449101670229b6741ff0a97d8439113c71f31530...,u_2f470606820175c7e14de69708bdbba617e582e998d7...
5,5796255,PBA-TIN-TI/2F,26568,u_74be240bf30daa8d0ac3b6c2a0b1fbce7a8af468d64b...,u_1724361ee5eef2cfaa30b4093d2dc88d092df5f3ac4a...
6,5796255,PBA-TIN-TI/2F,26568,u_b6040c5daba507e2861577ac6b35d37194c826db52b9...,u_606a0f586891007a50491ed738ad911d245ac5f18a7c...
7,5796255,PBA-TIN-TI/2F,26568,u_6d9bfc8d7fd7614a1ac079ef9072886a670fd457815d...,u_337355d0339c4a131f9d4fd4e30636d7192d3a610326...
8,5796255,PBA-TIN-TI/2F,26568,u_cb38e6456a55731c1b039ff84b5d618e191ec9a4586c...,u_7434b592bf25cf7ff2090849cb4a72b3d5378488b96d...
9,5796255,PBA-TIN-TI/2F,26568,u_4776953658839d283f693f34f43205cd8b8c22399c83...,u_e53ddfc06daf997681e33498e4720433e65436112ee6...
